# Progetto Elementi di Bio informatica

### Import librerie necessarie

In [2]:
import pysam
import pandas as pd
from collections import defaultdict

### Funzione per lettura GTF

In [3]:
def parse_gtf(gtf_file):
    exons = [] #Lista dove salverò gli esonoi letti
    with open(gtf_file) as f: #Apro il file GTF 
        for line in f: #Itero su ogni riga del GTF e la leggo 
            fields = line.strip().split('\t') #Divido in colonne anzi che la separazione con tab
            
            chrom, _, feature, start, end, _, _, _, attributes = fields #definisco la struttura dell'attributo
           
            attr_dict = {} #Creo un dizionario dove salvo gli attributi
            
            #Adatto il formato di ciascun attributo 
            for attr in attributes.split(';'):#Divide la stringa in attributi utilizzando ; come separatore
                parts = attr.strip().split(' ', 1) #vienen divisa ongi sottostinga in chiave valore rimuovendo spazi iniziali e finali
                if len(parts) == 2: #viene controllato che siano effettivamente due valori chiave valore
                    key, value = parts # diviede in due variabili il contenuto di parts
                    attr_dict[key] = value.strip('"') #aggiunge la coppia chiave valore al dizionario.
            exons.append({
                'chrom': chrom,
                'start': int(start),
                'end': int(end),
                'gene_id': attr_dict.get('gene_id'),
                'transcript_id': attr_dict.get('transcript_id')
            })
    return pd.DataFrame(exons)

In [4]:
#TEST DELLA FUNZIONE
gtf_path = "chr8/annotation_one_tr_chr8.gtf"

parse_gtf(gtf_path)

,chrom,start,end,gene_id,transcript_id
0,8,725777,725877,ENSG00000254207,ENST00000520760
1,8,725188,725402,ENSG00000254207,ENST00000520760
2,8,1299925,1302607,ENSG00000275427,ENST00000617093
3,8,1297033,1297228,ENSG00000275427,ENST00000617093
4,8,1296034,1296203,ENSG00000275427,ENST00000617093
...,...,...,...,...,...
11528,8,10050485,10054254,ENSG00000260093,ENST00000562143
11529,8,24387161,24387515,ENSG00000253535,ENST00000518988
11530,8,24300885,24300929,ENSG00000253535,ENST00000518988
11531,8,24296154,24296250,ENSG00000253535,ENST00000518988


### Funzione per estrarre M blocchi

In [5]:
# Funzione che estrae i blocchi M da un read
def get_aligned_blocks(read):
    blocks = [] #definisco una lista che mi contera le cordinate start end delle sottostringhe del read
    pos = read.reference_start # indica la posizione, inizilamente è l'inizio del read
    for op, length in read.cigartuples: # Itero per ogni informazione della CIGAR string e mi salvo le informazioni
        if op == 0:  # M
            blocks.append((pos, pos + length))
            pos += length
        elif op == 3:  # N
            pos += length
    return blocks

In [6]:
#TEST DELLA FUNZIONE 
class MockRead:
    def __init__(self, reference_start, cigartuples):
        self.reference_start = reference_start
        self.cigartuples = cigartuples

# Test: un read con due blocchi M separati da uno splicing (N)
read = MockRead(
    reference_start=100,
    cigartuples=[(0, 50), (3, 100), (0, 30)]  # M50 N100 M30
)

get_aligned_blocks(read)

[(100, 150), (250, 280)]

### Funzione per allineare i read agli esoni

In [7]:
# Funzione principale
def analyze_read_alignments(bam_path, exons_df):
    bamfile = pysam.AlignmentFile(bam_path, "rb") #Legge il bam in binario 
    
    exon_counts = defaultdict(int) #definisco un dizionrio che mi serve per definire quanti read si allineano con ciascun esone
    total_blocks = 0 #contatore per il conteggio dei blocchi allineati
   
    for read in bamfile: #itero su ongi read del file bam 

        chrom = bamfile.get_reference_name(read.reference_id)
        exons_chr = exons_df[exons_df["chrom"] == chrom]
        
        blocks = get_aligned_blocks(read) #ottengo le porzioni di read che si alllienano questo grazie alla CIGAR string
        total_blocks += len(blocks)

        for block_start, block_end in blocks: # per ongi blocco di esone trovato itero e vedo con qualie si allinea
            overlapping_exons = exons_chr[
                (exons_chr['start'] <= block_end) & (exons_chr['end'] >= block_start)
            ]
            for _, exon in overlapping_exons.iterrows(): # Se non esiste mi aggiunge al dizionario l'esone trovato, in caso mi somma +1
                exon_key = (exon['chrom'], exon['start'], exon['end'],
                            exon['transcript_id'], exon['gene_id'])
                exon_counts[exon_key] += 1

    bamfile.close()

    # Trova esone con più allineamenti
    if exon_counts:
        top_exon, top_count = max(exon_counts.items(), key=lambda x: x[1])
        top_exon_dict = {
            'chrom': top_exon[0],
            'start': top_exon[1],
            'end': top_exon[2],
            'transcript_id': top_exon[3],
            'gene_id': top_exon[4],
            'count': top_count
        }
    else:
        top_exon_dict = None

    return top_exon_dict, total_blocks

In [8]:
#TEST PER VERIFICARME IL FUNZIONAMENTO
gtf_path = "test.gtf"
bam_path = "test.bam"
#output_fastq = "spliced_reads_chr8.fastq"

exons_df = parse_gtf(gtf_path)
best_exon_info, total_blocks = analyze_read_alignments(bam_path, exons_df)

print(f"Numero totale di segmenti (blocchi M) analizzati: {total_blocks}")

if best_exon_info:
    print("Esone più coperto:")
    for k, v in best_exon_info.items():
        print(f"{k}: {v}")
else:
    print("Nessun esone coperto.")

Numero totale di segmenti (blocchi M) analizzati: 4
Esone più coperto:
chrom: chr1
start: 100
end: 200
transcript_id: tx1
gene_id: gene1
count: 2


[E::idx_find_and_load] Could not retrieve index file for 'test.bam'


In [9]:
#Funzione per stampare contenuto del file bam 
def print_bam_contents(bam_path, max_reads=10):
    """
    Stampa informazioni sui primi max_reads reads del file BAM.
    """
    with pysam.AlignmentFile(bam_path, "rb") as bamfile:
        count = 0
        for read in bamfile:
            print(f"Read name: {read.query_name}")
            print(f"Reference start: {read.reference_start}")
            print(f"CIGAR string: {read.cigarstring}")
            print(f"Sequence: {read.query_sequence}")
            print("-" * 40)
            count += 1
            if count >= max_reads:
                break


In [10]:
print_bam_contents(bam_path)

Read name: read1
Reference start: 110
CIGAR string: 12M
Sequence: ACTGACTGACTG
----------------------------------------
Read name: read2
Reference start: 255
CIGAR string: 12M
Sequence: TTGGAATTCCGG
----------------------------------------
Read name: read3
Reference start: 190
CIGAR string: 10M50N2M
Sequence: AAACCCGGGTTT
----------------------------------------


[E::idx_find_and_load] Could not retrieve index file for 'test.bam'


In [11]:
parse_gtf(gtf_path)

,chrom,start,end,gene_id,transcript_id
0,chr1,100,200,gene1,tx1
1,chr1,250,300,gene1,tx1
2,chr1,400,500,gene2,tx2


### Funzione per estrerre i read spliced e salvataggio in un file  FASTQ

In [12]:
#Funzione per estrarre tutti i read spliced e salvarli in un file fastq
def extract_spliced_to_fastq(bam_path, output_fastq_path):
    bamfile = pysam.AlignmentFile(bam_path, "rb")
    spliced_count = 0

    with open(output_fastq_path, "w") as out_f:
        for read in bamfile:
            if read.is_unmapped or read.cigarstring is None:
                continue
            if "N" in read.cigarstring:  # Se è spliced
                # Converti la qualità in formato FASTQ (Phred+33)
                qual_str = ''.join([chr(q + 33) for q in read.query_qualities]) if read.query_qualities else 'I' * len(read.query_sequence)
                # Scrivi il record in formato FASTQ
                out_f.write(f"@{read.query_name}\n{read.query_sequence}\n+\n{qual_str}\n")
                spliced_count += 1

    bamfile.close()
    return spliced_count

### Applico il codice al chr8

In [13]:
gtf_path = "chr8/annotation_one_tr_chr8.gtf"
bam_path = "chr8/sample_chr8.bam"
output_fastq = "spliced_reads_chr8.fastq"

exons_df = parse_gtf(gtf_path)
best_exon_info, total_blocks = analyze_read_alignments(bam_path, exons_df)

print(f"Numero totale di segmenti (blocchi M) analizzati: {total_blocks}")

if best_exon_info:
    print("Esone più coperto:")
    for k, v in best_exon_info.items():
        print(f"{k}: {v}")
else:
    print("Nessun esone coperto.")
    
n_spliced = extract_spliced_to_fastq(bam_path, output_fastq)
print(f"Read spliced salvati in FASTQ: {n_spliced}")

[E::idx_find_and_load] Could not retrieve index file for 'chr8/sample_chr8.bam'


Numero totale di segmenti (blocchi M) analizzati: 1648
Esone più coperto:
chrom: 8
start: 85134847
end: 85135032
transcript_id: ENST00000517875
gene_id: ENSG00000133739
count: 158
Read spliced salvati in FASTQ: 587


[E::idx_find_and_load] Could not retrieve index file for 'chr8/sample_chr8.bam'


### Applico il codice al chr21

In [14]:
gtf_path = "chr21/annotation_one_tr_chr21.gtf"
bam_path = "chr21/sample_chr21.bam"
output_fastq = "spliced_reads_chr21.fastq"

exons_df = parse_gtf(gtf_path)
best_exon_info, total_blocks = analyze_read_alignments(bam_path, exons_df)

print(f"Numero totale di segmenti (blocchi M) analizzati: {total_blocks}")

if best_exon_info:
    print("Esone più coperto:")
    for k, v in best_exon_info.items():
        print(f"{k}: {v}")
else:
    print("Nessun esone coperto.")

n_spliced = extract_spliced_to_fastq(bam_path, output_fastq)
print(f"Read spliced salvati in FASTQ: {n_spliced}")

[E::idx_find_and_load] Could not retrieve index file for 'chr21/sample_chr21.bam'


Numero totale di segmenti (blocchi M) analizzati: 752
Esone più coperto:
chrom: 21
start: 42098926
end: 42099180
transcript_id: ENST00000466434
gene_id: ENSG00000177398
count: 211
Read spliced salvati in FASTQ: 262


[E::idx_find_and_load] Could not retrieve index file for 'chr21/sample_chr21.bam'
